In [ ]:
import pandas as pd
import numpy as np

from sklearn.metrics.pairwise import (
    cosine_similarity, 
    euclidean_distances, 
    manhattan_distances, 
    cosine_distances, 
)

from config import (
    dataset_dir, 
    dataset_preprocessed_name, 
    dataset_cleaned_name, 
)

In [216]:
df = pd.read_pickle(dataset_dir / dataset_preprocessed_name)
df.head()

,vote_average_norm,release_year_norm,popularity_log_norm,vote_count_log_norm,language_code,genres_multihot,title_embeddings,original_title_embeddings,overview_embeddings
id,,,,,,,,,
472027,1.480033,0.483563,0.273992,-0.250526,11,"[1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, ...","[-0.01876905746757984, 0.025998396798968315, -...","[-0.01876901276409626, 0.025998389348387718, -...","[0.04105142503976822, 0.03513537347316742, -0...."
828898,0.710353,0.929414,0.103787,-0.553464,11,"[1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, ...","[-0.05961787328124046, 0.011813832446932793, -...","[-0.05961785465478897, 0.011813849210739136, -...","[-0.057032689452171326, 0.008622821420431137, ..."
394355,0.092334,0.305223,-0.440782,-0.031040,11,"[1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, ...","[-0.061722300946712494, 0.09019377827644348, -...","[-0.061722300946712494, 0.0901937484741211, -0...","[-0.04759906232357025, 0.04371899366378784, 0...."
640810,0.425988,0.349808,-0.701603,-1.039007,11,"[1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, ...","[-0.07855816930532455, 0.09684545546770096, -0...","[-0.07855820655822754, 0.09684544801712036, -0...","[-0.03096647560596466, 0.07531556487083435, -0..."
13204,-0.191083,0.216053,-0.193108,-0.281955,11,"[1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, ...","[-0.04970906674861908, 0.06194702908396721, -0...","[-0.04970905929803848, 0.061947036534547806, -...","[-0.04846826568245888, 0.060932304710149765, -..."


In [217]:
def calc_lp_norm(a, b, p: int = 1) -> float:
    d = sum([abs(a[i] - b[i]) ** p for i in range(len(a))])**(1 / p)
    return d

In [218]:
original_title_embeddings = df['original_title_embeddings'].to_numpy()[0]
title_embeddings = df['title_embeddings'].to_numpy()[0]

In [219]:
for p in range(1, 10):
    print(f'p={p}\t{calc_lp_norm(original_title_embeddings, title_embeddings, p)}')

p=1	7.193819330453266e-06
p=2	4.607769142879862e-07
p=3	2.0078760222276694e-07
p=4	1.3840522961852655e-07
p=5	1.1344089068961685e-07
p=6	1.0081533071846471e-07
p=7	9.351195060871307e-08
p=8	8.889305446739688e-08
p=9	8.577526228188335e-08


In [220]:
for metric in [cosine_similarity, cosine_distances, euclidean_distances, manhattan_distances]:
    distances = metric(
        original_title_embeddings.reshape(1, -1), 
        title_embeddings.reshape(1, -1),
    )
    print(f"{metric.__name__:20}: {distances[0][0]}")

cosine_similarity   : 0.9999999999998943
cosine_distances    : 1.056932319443149e-13
euclidean_distances : 4.5976783694450593e-07
manhattan_distances : 7.193819330453266e-06


In [221]:
def find_neighbors(
        index: int, 
        column: pd.Series, 
        metric: str = 'euclidean_distances', 
        k: int = 3
    ) -> np.ndarray:
    base = column[index]
    distances = []

    metrics = [cosine_similarity, cosine_distances, euclidean_distances, manhattan_distances]
    metrics = {metric.__name__: metric for metric in metrics}

    metric_func = metrics[metric]

    for x in column:
        distances.append(
            metric_func(
                base.reshape(1, -1), x.reshape(1, -1)
            )[0][0]
        )
    
    top_k = np.argsort(distances)[:k + 1]

    return top_k

In [222]:
column = df['title_embeddings'].to_numpy()
df_original = pd.read_pickle(dataset_dir / dataset_cleaned_name)

In [223]:
neighbors = find_neighbors(index=10, column=column, metric='cosine_distances')
df_original.iloc[neighbors]

,title,original_title,overview,original_language,genres,popularity,vote_average,vote_count,release_year
id,,,,,,,,,
1010830,Batman: The Long Halloween Deluxe Edition,Batman: The Long Halloween Deluxe Edition,Atrocious serial killings on holidays in Gotha...,en,"[Animation, Action, Thriller, Crime, Mystery]",2.5232,7.100,54,2022
736073,"Batman: The Long Halloween, Part One","Batman: The Long Halloween, Part One",Following a brutal series of murders taking pl...,en,"[Animation, Mystery, Action, Crime]",6.3382,7.445,723,2021
736074,"Batman: The Long Halloween, Part Two","Batman: The Long Halloween, Part Two","As Gotham City's young vigilante, the Batman, ...",en,"[Animation, Mystery, Action, Crime]",5.9403,7.397,594,2021
806704,The Batman: Part II,The Batman: Part II,Sequel to the 2022 film The Batman.,en,"[Mystery, Drama, Crime]",6.1311,0.000,0,2028


In [224]:
neighbors = find_neighbors(index=10, column=column, metric='manhattan_distances')
df_original.iloc[neighbors]

,title,original_title,overview,original_language,genres,popularity,vote_average,vote_count,release_year
id,,,,,,,,,
1010830,Batman: The Long Halloween Deluxe Edition,Batman: The Long Halloween Deluxe Edition,Atrocious serial killings on holidays in Gotha...,en,"[Animation, Action, Thriller, Crime, Mystery]",2.5232,7.100,54,2022
736073,"Batman: The Long Halloween, Part One","Batman: The Long Halloween, Part One",Following a brutal series of murders taking pl...,en,"[Animation, Mystery, Action, Crime]",6.3382,7.445,723,2021
736074,"Batman: The Long Halloween, Part Two","Batman: The Long Halloween, Part Two","As Gotham City's young vigilante, the Batman, ...",en,"[Animation, Mystery, Action, Crime]",5.9403,7.397,594,2021
806704,The Batman: Part II,The Batman: Part II,Sequel to the 2022 film The Batman.,en,"[Mystery, Drama, Crime]",6.1311,0.000,0,2028
